In [ ]:
import requests
from autogen_agentchat.messages import TextMessage, MultiModalMessage
from autogen_core import Image as AGImage
from PIL import Image
from dotenv import load_dotenv
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from typing import Literal


In [ ]:
load_dotenv(override=True)

In [ ]:
url = "/Users/testmachine/Downloads/test_image.png"

pil_image = Image.open(url)
image = AGImage(pil_image)

image

In [ ]:
multimodal_message = MultiModalMessage(content=["Elaborate the content of the image in detail", image], source="User")

In [ ]:
#client = OllamaChatCompletionClient(model="llama3.2")
client = OpenAIChatCompletionClient(model="gpt-4o-mini")

detailer = AssistantAgent(name="detailer_agent",
    model_client=client, system_message="You are expert in detailing about images")

In [ ]:
response = await detailer.on_messages([multimodal_message], cancellation_token=CancellationToken())
reply = response.chat_message.content
display(Markdown(reply))

In [ ]:
#Structured Output

class ImageDetails(BaseModel):
    scene: str = Field(description="Describe the scene of image")
    message: str = Field(description="What image is trying to convey")
    style: str = Field(description="Artistic style of the image")
    orientation: Literal["portrait", "landscape", "square"] = Field(description="Orientation of the image")

In [ ]:
#client = OllamaChatCompletionClient(model="llama3.2")
client = OpenAIChatCompletionClient(model="gpt-4o-mini")

detailer = AssistantAgent(name="detailer_agent",
    model_client=client, system_message="You are expert in detailing about images",
    output_content_type=ImageDetails)

In [ ]:
response = await detailer.on_messages([multimodal_message], cancellation_token=CancellationToken())
reply = response.chat_message.content
reply

In [ ]:
import textwrap

print(f"Scene:\n{textwrap.fill(reply.scene)}\n\n")
print(f"Message:\n{textwrap.fill(reply.message)}\n\n")
print(f"Style:\n{textwrap.fill(reply.style)}\n\n")
print(f"Orientation:\n{textwrap.fill(reply.orientation)}\n\n")